
# **Lab 4: Predicting Delays (Regression with BQML)**
**Unit 2 • Week 7 (Thu) — ML Foundations & Regression**

**Objective:** Train, evaluate, and interpret a **linear regression** model with **BigQuery ML** to predict a flight’s **arrival delay (minutes)**. You’ll use Gemini for feature brainstorming and explanation, and run all SQL from **Colab**.

**Business Goal:** Improve airline resource planning by predicting arrival delays.

> **Dataset (recommended):** `bigquery-public-data.airline_ontime_data.flights`  
> If your project uses a different on-time dataset, update the `FULL_TABLE` constant below accordingly.


## Setup & Authentication

In [ ]:
# Authenticate and initialize BigQuery client
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
import pandas as pd

# ▶️ Set your GCP project id
PROJECT_ID = "big-data-analysis-472319"

# ▶️ Table containing flights data (update if your dataset differs)
FULL_TABLE = "bigquery-samples.airline_ontime_data.flights"  # e.g., "your_project.your_dataset.flights"

client = bigquery.Client(project=PROJECT_ID)
print("Authenticated. Project:", PROJECT_ID)
print("Using table:", FULL_TABLE)

Authenticated. Project: big-data-analysis-472319
Using table: bigquery-samples.airline_ontime_data.flights


In [ ]:
# Query the head of the table
query = f"SELECT * FROM `{FULL_TABLE}` LIMIT 5"
df_head = client.query(query).to_dataframe()
display(df_head)

,date,airline,airline_code,departure_airport,departure_state,departure_lat,departure_lon,arrival_airport,arrival_state,arrival_lat,arrival_lon,departure_schedule,departure_actual,departure_delay,arrival_schedule,arrival_actual,arrival_delay
0,2005-09-05,EV,20366,CVG,KY,39.04,-84.66,ABE,PA,40.65,-75.44,1100,1100,0.0,1234,1220,-14.0
1,2008-07-22,EV,20366,CVG,KY,39.04,-84.66,ABE,PA,40.65,-75.44,1945,2002,17.0,2124,2123,-1.0
2,2005-08-01,EV,20366,CVG,KY,39.04,-84.66,ABE,PA,40.65,-75.44,1050,1048,-2.0,1224,1213,-11.0
3,2005-08-03,EV,20366,CVG,KY,39.04,-84.66,ABE,PA,40.65,-75.44,1050,1048,-2.0,1224,1211,-13.0
4,2004-12-14,EV,20366,CVG,KY,39.04,-84.66,ABE,PA,40.65,-75.44,1847,1903,16.0,2014,2033,19.0



---
## Feature Brainstorming (Gemini Prompt)

Use Gemini to brainstorm predictive features:

```python
prompt = # TASK: Brainstorm features for a machine learning model.
# CONTEXT: I'm using the BigQuery public flights dataset. I want to predict the 'arr_delay' (arrival delay in minutes), which is a numerical value.
# GOAL: List 5 columns from the dataset that you think would be the best predictors for 'arr_delay' and briefly explain why for each one.
```
Add your chosen features below.


**Chosen features:** `departure_schedule`, `arrival_schedule`, `arrival_delay`, `arrival_state`, `departure_state`


---
## Train Your First Regression Model

Use **BQML** to create a **linear regression** model. Edit the feature list as needed.  
We also enable explanations to support **ML.EXPLAIN_PREDICT** and global feature importance.


In [4]:
# Create a linear regression model
# NOTE: Adjust the feature list if your columns differ.
model_id = f"{PROJECT_ID}.superstore_data.flight_delay_predictor"  # You may change dataset/model name

create_model_sql = f"""
CREATE OR REPLACE MODEL `{PROJECT_ID}.superstore_data.flight_delay_predictor`
OPTIONS(
  model_type='linear_reg',
  input_label_cols=['arrival_delay'],
  enable_global_explain=TRUE
) AS
SELECT
  CAST(arrival_delay AS FLOAT64) AS arrival_delay,
  CAST(departure_schedule AS FLOAT64) AS departure_schedule,
  CAST(arrival_schedule AS FLOAT64) AS arrival_schedule,
  arrival_state,
  departure_state,
  airline
FROM `{FULL_TABLE}`
WHERE arrival_delay IS NOT NULL
LIMIT 200000;  -- keep costs low
"""

job = client.query(create_model_sql)
job.result()
print("Model created:", model_id)

Model created: big-data-analysis-472319.superstore_data.flight_delay_predictor



---
## Evaluate Model Performance (`ML.EVALUATE`) — Validate

Run evaluation to view **mean_absolute_error**, **r2_score**, etc. Then use Gemini to explain a key metric to a non-technical audience.

**Gemini Explainer Prompt:**

```python
prompt = """
TASK: Explain a machine learning evaluation metric.
CONTEXT: I have trained a linear regression model using BigQuery ML to predict flight arrival delays ('arrival_delay'). I have the evaluation results from ML.EVALUATE, and one of the metrics is Mean Absolute Error (MAE). The MAE value is {mae_value}.
GOAL: Explain the Mean Absolute Error (MAE) metric in simple terms to a non-technical audience, using the provided MAE value.
"""
```


In [5]:
eval_sql = f"SELECT * FROM ML.EVALUATE(MODEL `{PROJECT_ID}.superstore_data.flight_delay_predictor`)"
eval_df = client.query(eval_sql).result().to_dataframe()
eval_df

,mean_absolute_error,mean_squared_error,mean_squared_log_error,median_absolute_error,r2_score,explained_variance
0,20.337275,1174.480919,2.526187,13.891573,0.040027,0.040043


**Interpretation**: The model’s overall predictive performance is poor, with high average errors and low explanatory power. It likely needs additional or better-quality features to capture the patterns influencing arrival delays.




---
## Explain Predictions (`ML.EXPLAIN_PREDICT`) — Why did the model predict that?

Author your own prompt to generate a query that explains a **hypothetical** case:  
> Predict the arrival delay for a 2000-mile flight with a 30-minute departure delay on carrier 'AA', and explain the top contributing features.

**Hint:** You’ll need a small input table with the required features.

> **Your Gemini prompt (write in the next cell):**  
> *Ask for a full `ML.EXPLAIN_PREDICT` example for this model and scenario, returning top features.*


In [7]:
explain_sql = f"""
# Use ML.EXPLAIN_PREDICT on the trained model
SELECT
  *
FROM
  ML.EXPLAIN_PREDICT(
    MODEL `{PROJECT_ID}.superstore_data.flight_delay_predictor`, # Specify the model
    (
      SELECT
        # Create an inline table with hypothetical flight details using model features
        -- Example values for a hypothetical flight:
        1400 AS departure_schedule, -- e.g., 2:00 PM
        1600 AS arrival_schedule,   -- e.g., 4:00 PM
        'CA' AS arrival_state,
        'NY' AS departure_state,
        'AA' AS airline
    ),
    STRUCT(3 AS top_k_features) # Request the top 3 contributing features
  )
"""
explain_df = client.query(explain_sql).result().to_dataframe()
explain_df

,predicted_arrival_delay,top_feature_attributions,baseline_prediction_value,prediction_value,approximation_error,departure_schedule,arrival_schedule,arrival_state,departure_state,airline
0,5.908416,"[{'feature': 'arrival_state', 'attribution': -...",15406.206678,5.908416,0.0,1400,1600,CA,NY,AA


Intepretation: The model predicts a moderate delay (~6 minutes) for this flight, primarily influenced by route-related factors such as the departure and arrival states. However, since the model’s r2_score is very low, this prediction should be treated with low confidence, as it captures only weak patterns in the data.


---
## ✅ Deliverable for Lab 4

- Completed `Lab4_Regression_BQML.ipynb` with:
  - Brainstormed features + rationale
  - `CREATE MODEL` SQL
  - `ML.EVALUATE` output and explanation
  - `ML.EXPLAIN_PREDICT` output (hypothetical case)
- Push to **GitHub** and submit the link on **Brightspace**.
